In [ ]:
import networkx as nx
from pathlib import Path
from dash import Dash, html
import dash_cytoscape as cyto

# Paths for Mac
# file1 = "/Users/nickpellegri/Documents/Github/w0rldview/test1.graphml"
# file2 = "/Users/nickpellegri/Documents/Github/w0rldview/test2.graphml"

# Paths for Windows
file1 = Path(r"C:\WorldView\worldview\w0rldview\test nodes\node_1.graphml")
file2 = Path(r"C:\WorldView\worldview\w0rldview\test nodes\node_2.graphml")

def load_and_make_elements(path, seed=42):
    G = nx.read_graphml(path)
    pos = nx.spring_layout(G, seed=seed)
    # scale/translate positions for Cytoscape (pixels)
    scale = 700
    offset = 350
    def to_pos(p):
        return {"x": float(p[0] * scale + offset), "y": float(p[1] * scale + offset)}

    nodes = []
    for n in G.nodes():
        nodes.append({
            "data": {"id": str(n), "label": str(n)},
            "position": to_pos(pos[n])
        })

    edges = []
    for u, v in G.edges():
        edges.append({
            "data": {"id": f"{u}__{v}", "source": str(u), "target": str(v)}
        })

    return G, nodes, edges

# Load both graphs
G1, nodes1, edges1 = load_and_make_elements(file1, seed=42)
G2, nodes2, edges2 = load_and_make_elements(file2, seed=7)  # different seed for variation

# Determine matching vs unique
nset1 = set(G1.nodes())
nset2 = set(G2.nodes())
eset1 = set(G1.edges())
eset2 = set(G2.edges())

matching_nodes = nset1 & nset2
matching_edges = eset1 & eset2

def mark_elements(nodes, edges, matching_nodes, matching_edges):
    out = []
    for n in nodes:
        n_id = n["data"]["id"]
        cls = "match" if n_id in matching_nodes else "unique"
        n["classes"] = cls
        out.append(n)
    for e in edges:
        s = e["data"]["source"]
        t = e["data"]["target"]
        cls = "match" if ( (s, t) in matching_edges or (s, t) in {(b,a) for (a,b) in matching_edges} ) else "unique"
        e["classes"] = cls
        # add arrow shape on edges
        e["data"]["directed"] = True
        out.append(e)
    return out

elements1 = mark_elements(nodes1, edges1, matching_nodes, matching_edges)
elements2 = mark_elements(nodes2, edges2, matching_nodes, matching_edges)

# Cytoscape stylesheet
stylesheet = [
    {"selector": "node", "style": {"label": "data(label)", "width": 200, "height": 200, "font-size": 10,
                                   "text-valign": "center", "text-halign": "center",
                                   "border-width": 1, "border-color": "#222"}},
    {"selector": "edge", "style": {"width": 3, "line-color": "#bbb", "target-arrow-color": "#bbb",
                                   "target-arrow-shape": "triangle", "curve-style": "bezier"}},
    {"selector": ".match", "style": {"background-color": "#bfc9d9", "line-color": "#bfc9d9", "target-arrow-color": "#bfc9d9", "opacity": 0.7}},
    {"selector": ".unique", "style": {"background-color": "#ff6b6b", "line-color": "#ff6b6b", "target-arrow-color": "#ff6b6b", "opacity": 1, "line-style": "dashed"}},
]

app = Dash(__name__)

app.layout = html.Div([
    html.Div([
        html.H3(file1.name, style={"textAlign": "center", "margin": "6px"}),
        cyto.Cytoscape(
            id="cytoscape-1",
            elements=elements1,
            stylesheet=stylesheet,
            style={"width": "48vw", "height": "80vh", "display": "inline-block"},
            layout={"name": "preset"}
        )
    ], style={"display": "inline-block", "verticalAlign": "top", "width": "49%"}),

    html.Div([
        html.H3(file2.name, style={"textAlign": "center", "margin": "6px"}),
        cyto.Cytoscape(
            id="cytoscape-2",
            elements=elements2,
            stylesheet=stylesheet,
            style={"width": "48vw", "height": "80vh", "display": "inline-block"},
            layout={"name": "preset"}
        )
    ], style={"display": "inline-block", "verticalAlign": "top", "width": "49%"}),
], style={"display": "flex", "justifyContent": "space-between", "padding": "8px"})

if __name__ == "__main__":
    app.run(port=8060, debug=True)

Exception in thread Thread-82 (<lambda>):
Traceback (most recent call last):
  File "c:\Users\NP232582\AppData\Local\anaconda3\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\NP232582\AppData\Local\anaconda3\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "c:\Users\NP232582\AppData\Local\anaconda3\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\NP232582\AppData\Local\anaconda3\Lib\site-packages\dash\dash.py", line 2185, in <lambda>
    target=lambda: _watch.watch(
                   ~~~~~~~~~~~~^
        [self.config.assets_folder] + component_packages_dist,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self._on_assets_change,
        ^^^^^^^^^^^^^^^^^^^^^^^
        sleep_time=dev_tools.hot_reload_watch_interval,
        ^^^^^^^^